# BLIP + LoRA on ROCOv2 — Complete Notebook with Network Checks and Local Data Support

This notebook includes all code for data preparation, training, LoRA checkpoint saving, caption generation, and evaluation. The original experiment settings are retained: 10% of the training data, six epochs, batch size 3, and the original LoRA configuration.

**If you encounter `Temporary failure in name resolution`:** enable Internet in the Kaggle notebook settings and rerun the connectivity check. This error concerns network access; Python code cannot enable a disabled Internet connection in the runtime.

For offline execution, attach the dataset using **Add Input** and set `DATA_MODE="local"`. Offline training also requires the necessary packages and a **complete pretrained BLIP model** to be installed or available locally. Dataset files alone are not sufficient.

Open this notebook in a **fresh kernel**, select a GPU, and execute the cells in order. If you already ran a previous version in the same kernel, restart it first to remove previously imported modules from memory.

The complete GPU training run and final caption quality have not been evaluated in the environment used to prepare this notebook. A tiny-model CPU integration check covers training and adapter reload; separate checks cover local files and DNS failures.


## 1. Settings and Paths

The `auto` mode first checks for existing data under `/kaggle/input` and `/kaggle/working`. When captions and either extracted images or local image archives are available, that split does not require a Zenodo request. You can also provide the four optional explicit data paths.

For online loading, `MODEL_SOURCE` is the Hugging Face model ID. For offline loading, set it to a directory containing the full pretrained weights, `config.json`, `preprocessor_config.json`, and tokenizer files. Choose a new `RUN_NAME` for each fresh training experiment.


In [1]:
from pathlib import Path
import sys

WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
DATA_ROOT = WORK_ROOT
IMAGES_ROOT = WORK_ROOT / "datasets"
RUN_NAME = "blip-corrected-network-v2-10pct-6epochs"

# auto: reuse local data first, download missing splits only when possible.
# local: never download data; use Kaggle inputs or existing local archives.
DATA_MODE = "auto"
SEARCH_ROOTS = [Path("/kaggle/input"), WORK_ROOT]

# Optional explicit overrides. Leave None for automatic local discovery.
LOCAL_TRAIN_CSV = None
LOCAL_TEST_CSV = None
LOCAL_TRAIN_IMAGES_DIR = None
LOCAL_TEST_IMAGES_DIR = None

# For offline model loading, set this to a FULL pretrained BLIP directory.
MODEL_SOURCE = "Salesforce/blip-image-captioning-base"

# auto: reuse installed packages when available; install missing ones if online.
# force: install the pinned dependency profile for this Python version.
# skip: do not install anything (use a prepared offline environment).
INSTALL_DEPENDENCIES = "auto"

RUN_SANITY_CHECK = True
RUN_TRAINING = True
RUN_EVALUATION = True
RUN_ZERO_SHOT_BASELINE = False

CODE_ROOT = WORK_ROOT / "blip_corrected_code_v2"
CODE_ROOT.mkdir(parents=True, exist_ok=True)
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))
print("Python:", sys.version.split()[0])
print("Code:", CODE_ROOT)
print("Data mode:", DATA_MODE)


Python: 3.13.15
Code: /kaggle/working/blip_corrected_code_v2
Data mode: auto


## 2. Connectivity Check

This cell reports access to the three required services. If Zenodo is unavailable but the local dataset is complete, data preparation can continue. A complete cached BLIP model can also be loaded without online access to Hugging Face.


In [2]:
import socket
import urllib.request
import urllib.error
from concurrent.futures import ThreadPoolExecutor

def network_probe(item):
    label, url = item
    try:
        request = urllib.request.Request(url, headers={"User-Agent": "BLIP-notebook-diagnostic"})
        with urllib.request.urlopen(request, timeout=8) as response:
            response.read(1)
        return label, {"ok": True, "detail": "reachable"}
    except Exception as error:
        reason = getattr(error, "reason", error)
        kind = "DNS/name resolution" if isinstance(reason, socket.gaierror) else type(error).__name__
        return label, {"ok": False, "detail": f"{kind}: {reason}"}

checks = [
    ("PyPI", "https://pypi.org/pypi/transformers/json"),
    ("Hugging Face", "https://huggingface.co/api/models/Salesforce/blip-image-captioning-base"),
    ("Zenodo", "https://zenodo.org/api/records/10821435"),
]
with ThreadPoolExecutor(max_workers=3) as executor:
    NETWORK_REPORT = dict(executor.map(network_probe, checks))
for label, status in NETWORK_REPORT.items():
    print(f"{label}: {'OK' if status['ok'] else 'UNAVAILABLE'} — {status['detail']}")
if not any(status['ok'] for status in NETWORK_REPORT.values()):
    print("Internet/DNS appears unavailable. Enable Internet in Kaggle settings, or attach both the dataset and full pretrained model for local execution.")
print("These are diagnostics only; complete local files can be used without network access.")


PyPI: OK — reachable
Hugging Face: OK — reachable
Zenodo: OK — reachable
These are diagnostics only; complete local files can be used without network access.


## 3. Dependencies

The default mode reuses installed packages and installs dependencies only when required packages are missing. The installation profile for Python 3.13 differs from the profile for Python 3.12 and earlier. Kaggle's existing PyTorch installation is retained.

If the installation cell asks you to restart the kernel, do so, keep `INSTALL_DEPENDENCIES="auto"`, and execute the notebook again from the beginning.


## 4. Complete Project Source

Run every cell in this section. These cells write the required Python modules from the source code embedded in this notebook. No ZIP download or repository clone is required.


In [3]:
import importlib.util
import importlib.metadata
import subprocess
import sys

required_modules = ["torch", "transformers", "peft", "accelerate", "numpy", "pandas", "PIL", "yaml", "tqdm", "pycocoevalcap"]
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]

if INSTALL_DEPENDENCIES not in ("auto", "force", "skip"):
    raise ValueError("INSTALL_DEPENDENCIES must be auto, force, or skip.")

if INSTALL_DEPENDENCIES == "skip":
    if missing:
        raise RuntimeError(f"Offline environment is missing packages: {missing}. Enable Internet and use auto, or attach a prepared wheel collection.") from None
    print("Using installed dependencies; installation skipped.")
elif INSTALL_DEPENDENCIES == "force" or missing:
    if not NETWORK_REPORT["PyPI"]["ok"]:
        raise RuntimeError(f"Cannot install dependencies because PyPI is unavailable. Missing packages: {missing}. Enable Kaggle Internet and rerun this cell. Fully offline execution also requires preinstalled packages or compatible attached wheels.") from None
    if "torch" in missing:
        raise RuntimeError("PyTorch is missing. Select a Kaggle GPU runtime that includes CUDA-enabled PyTorch before continuing.") from None
    if sys.version_info >= (3, 13):
        profile = ["transformers==4.57.6", "peft==0.18.1", "accelerate==1.12.0", "numpy>=2.1,<3", "pandas>=2.2.3,<3", "Pillow>=11,<12"]
    else:
        profile = ["transformers==4.44.2", "peft==0.12.0", "accelerate==0.33.0", "numpy>=1.26,<2", "pandas>=2.1,<3", "Pillow>=10,<12"]
    profile += ["PyYAML>=6,<7", "tqdm>=4.66,<5", "pycocoevalcap==1.2"]
    loaded_before = any(name in sys.modules for name in ("numpy", "transformers", "peft", "pandas"))
    result = subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q", *profile])
    if result.returncode:
        raise RuntimeError("Dependency installation failed. Inspect the pip output, check Internet, and start a fresh kernel before retrying.") from None
    if loaded_before:
        raise RuntimeError("Installation completed, but packages had already been imported. Restart the kernel, keep INSTALL_DEPENDENCIES='auto', and run this notebook again.") from None
    print("Dependencies installed. Existing CUDA-enabled PyTorch was retained.")
else:
    print("All required packages are already present; reusing them without a network installation.")

for distribution in ("torch", "transformers", "peft", "accelerate", "numpy", "pandas", "pycocoevalcap"):
    try:
        print(distribution, importlib.metadata.version(distribution))
    except importlib.metadata.PackageNotFoundError:
        pass


All required packages are already present; reusing them without a network installation.
torch 2.11.0+cu128
transformers 4.57.6
peft 0.18.1
accelerate 1.12.0
numpy 2.1.3
pandas 2.3.3
pycocoevalcap 1.2


### `src/__init__.py`

In [4]:
source = r'''"""Corrected BLIP + LoRA training and evaluation pipeline."""
'''
module_path = CODE_ROOT / 'src/__init__.py'
module_path.parent.mkdir(parents=True, exist_ok=True)
module_path.write_text(source, encoding='utf-8')


62

### `src/data/__init__.py`

In [5]:
source = r'''"""ROCOv2 data loading."""
'''
module_path = CODE_ROOT / 'src/data/__init__.py'
module_path.parent.mkdir(parents=True, exist_ok=True)
module_path.write_text(source, encoding='utf-8')


27

### `src/data/dataset.py`

In [6]:
source = r'''"""Load original PIL images once; let BlipProcessor resize and normalize."""

from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset


def validate_dataframe(dataframe):
    missing = {"ID", "Caption"} - set(dataframe.columns)
    if missing:
        raise ValueError(f"Missing CSV columns: {sorted(missing)}")
    if dataframe.empty:
        raise ValueError("The captions dataframe is empty.")
    if dataframe[["ID", "Caption"]].isna().any().any():
        raise ValueError("CSV IDs and captions must not contain missing values.")
    if dataframe["ID"].astype(str).duplicated().any():
        raise ValueError("Expected one unique ID per image in this ROCOv2 experiment.")
    if dataframe["Caption"].astype(str).str.strip().eq("").any():
        raise ValueError("Empty reference captions are not allowed.")


def image_path_for_id(images_dir, image_id):
    image_id = str(image_id)
    if "/" in image_id or "\\" in image_id or image_id in ("", ".", ".."):
        raise ValueError(f"Expected an image filename, got: {image_id!r}")
    filename = image_id if Path(image_id).suffix else image_id + ".jpg"
    return Path(images_dir) / filename


class MergeImageAndCaption(Dataset):
    """The original image/caption join, with IDs retained for evaluation."""

    def __init__(self, dataframe, images_dir):
        validate_dataframe(dataframe)
        self.dataset = dataframe.reset_index(drop=True).copy()
        self.dataset["ID"] = self.dataset["ID"].astype(str)
        self.image_list = [image_path_for_id(images_dir, value) for value in self.dataset["ID"]]
        missing = [str(path) for path in self.image_list if not path.is_file()]
        if missing:
            raise FileNotFoundError(
                f"{len(missing)} images are missing. Check the extracted train/test folder. "
                f"First missing file: {missing[0]}"
            )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        row = self.dataset.iloc[idx]
        with Image.open(self.image_list[idx]) as image:
            image = image.convert("RGB")
        return {"image": image, "text": str(row["Caption"]), "image_id": str(row["ID"])}


class ImageCaptioningDataset(Dataset):
    def __init__(self, dataset, processor, max_text_length=512):
        self.dataset = dataset
        self.processor = processor
        self.max_text_length = max_text_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        # Do not apply ToTensor()/divide by 255 before BlipProcessor.
        encoding = self.processor(
            images=item["image"],
            text=item["text"],
            return_tensors="pt",
            padding=False,
            truncation=True,
            max_length=self.max_text_length,
        )
        result = {key: value.squeeze(0) for key, value in encoding.items()}
        result["text"] = item["text"]
        result["image_id"] = item["image_id"]
        return result


def collate_fn(batch, processor):
    """Pad only to the longest caption in this batch and mask padding loss."""
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    input_ids = pad_sequence(
        [item["input_ids"] for item in batch], batch_first=True,
        padding_value=processor.tokenizer.pad_token_id,
    )
    attention_mask = pad_sequence(
        [item["attention_mask"] for item in batch], batch_first=True, padding_value=0,
    )
    labels = input_ids.clone()
    labels.masked_fill_(attention_mask.eq(0), -100)
    return {
        "pixel_values": pixel_values,
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "texts": [item["text"] for item in batch],
        "image_ids": [item["image_id"] for item in batch],
    }
'''
module_path = CODE_ROOT / 'src/data/dataset.py'
module_path.parent.mkdir(parents=True, exist_ok=True)
module_path.write_text(source, encoding='utf-8')


3950

### `src/data/download.py`

In [7]:
source = r'''"""Download only the four ROCOv2 files needed by this experiment."""

import argparse
import hashlib
import json
import socket
import time
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

RECORD_URL = "https://zenodo.org/api/records/10821435"


def open_url_with_retry(request, timeout=60, attempts=3):
    for attempt in range(attempts):
        try:
            return urllib.request.urlopen(request, timeout=timeout)
        except urllib.error.HTTPError as error:
            if error.code not in (408, 429, 500, 502, 503, 504):
                raise RuntimeError(
                    f"HTTP {error.code} while contacting Zenodo. "
                    "Check access to zenodo.org, or attach ROCOv2 files to Kaggle and use DATA_MODE='local'."
                ) from None
            last_error = str(error)
        except (urllib.error.URLError, socket.gaierror, TimeoutError, OSError) as error:
            last_error = str(error)
        if attempt + 1 < attempts:
            time.sleep(1 + attempt)
    raise RuntimeError(
        "Cannot reach Zenodo from this notebook. DNS/Internet may be unavailable. "
        "Enable Internet in Kaggle notebook settings and rerun this cell, or attach "
        "the dataset under /kaggle/input and use DATA_MODE='local'. "
        f"Last network error: {last_error}"
    ) from None


def md5_matches(path, expected):
    digest = hashlib.md5()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest() == expected


def download_file(info, destination):
    checksum = info.get("checksum", "")
    if not checksum.startswith("md5:"):
        raise ValueError("Expected an MD5 checksum in the official Zenodo manifest.")
    checksum = checksum.split(":", 1)[1]
    destination = Path(destination)
    if destination.is_file() and md5_matches(destination, checksum):
        print(f"Verified existing {destination.name}")
        return
    url = info["links"].get("self") or info["links"]["content"]
    request = urllib.request.Request(url, headers={"User-Agent": "BLIP-ROCOv2-research"})
    temporary = destination.with_suffix(destination.suffix + ".part")
    with open_url_with_retry(request, timeout=120) as response, temporary.open("wb") as handle:
        with tqdm(total=info.get("size"), unit="B", unit_scale=True, desc=destination.name) as bar:
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break
                handle.write(chunk)
                bar.update(len(chunk))
    if not md5_matches(temporary, checksum):
        raise RuntimeError(f"Checksum mismatch for {destination.name}; rerun the download.")
    temporary.replace(destination)


def safe_extract(archive_path, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(f"Archive member escapes destination: {member.filename}")
        archive.extractall(destination)


def download_and_extract(data_root="./data", images_root=None, splits=("train", "test")):
    data_root = Path(data_root)
    images_root = Path(images_root) if images_root is not None else data_root
    data_root.mkdir(parents=True, exist_ok=True)
    request = urllib.request.Request(RECORD_URL, headers={"User-Agent": "BLIP-ROCOv2-research"})
    with open_url_with_retry(request, timeout=60) as response:
        manifest = json.load(response)
    files = {entry["key"]: entry for entry in manifest["files"]}
    for split in splits:
        captions_path = data_root / f"{split}_captions.csv"
        download_file(files[captions_path.name], captions_path)
        ids = pd.read_csv(captions_path, dtype={"ID": str})["ID"]
        destination = images_root / f"{split}_ROCOv2"
        images_dir = destination / split
        if all((images_dir / (value if Path(value).suffix else value + ".jpg")).is_file() for value in ids):
            print(f"All {split} images already exist in {images_dir}; skipping image download.")
            continue
        archive_path = data_root / f"{split}_images.zip"
        download_file(files[archive_path.name], archive_path)
        print(f"Extracting {archive_path.name} ...")
        safe_extract(archive_path, destination)
    print("Data preparation complete. Check the CSV/image paths in your configuration.")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-root", default="./data")
    parser.add_argument("--images-root")
    args = parser.parse_args()
    download_and_extract(args.data_root, args.images_root)
'''
module_path = CODE_ROOT / 'src/data/download.py'
module_path.parent.mkdir(parents=True, exist_ok=True)
module_path.write_text(source, encoding='utf-8')


4981

### `src/data/prepare.py`

In [8]:
source = r'''"""Local-first ROCOv2 preparation and explicit online/offline model checks."""

import json
import urllib.error
import urllib.request
from pathlib import Path

import pandas as pd

from src.data.dataset import image_path_for_id, validate_dataframe
from src.data.download import download_and_extract, safe_extract


def probe_url(url, timeout=8):
    try:
        request = urllib.request.Request(url, headers={"User-Agent": "BLIP-notebook-check"})
        with urllib.request.urlopen(request, timeout=timeout) as response:
            response.read(1)
        return True, "reachable"
    except Exception as error:
        return False, f"{type(error).__name__}: {error}"


def unique_paths(paths):
    result, seen = [], set()
    for value in paths:
        if value is None:
            continue
        path = Path(value)
        key = str(path.resolve())
        if key not in seen:
            result.append(path)
            seen.add(key)
    return result


def files_named(roots, filename):
    found = []
    for root in roots:
        if root.is_dir():
            found.extend(sorted(root.rglob(filename)))
    return unique_paths(found)


def complete_images(folder, ids):
    folder = Path(folder)
    return folder.is_dir() and all(image_path_for_id(folder, value).is_file() for value in ids)


def find_images(split, ids, cfg, csv_path, roots, images_root):
    configured = Path(cfg["paths"][f"{split}_images_dir"])
    folders = [
        configured, Path(images_root) / f"{split}_ROCOv2" / split,
        csv_path.parent / split, csv_path.parent / f"{split}_ROCOv2" / split,
        csv_path.parent / "images" / split,
    ]
    for folder in unique_paths(folders):
        if complete_images(folder, ids):
            return folder
    filename = image_path_for_id(".", ids.iloc[0]).name
    for image in files_named(roots, filename):
        if complete_images(image.parent, ids):
            return image.parent
    return None


def resolve_split(split, cfg, roots, data_root, images_root):
    csv_candidates = unique_paths([
        cfg["paths"][f"{split}_csv"], Path(data_root) / f"{split}_captions.csv",
        *files_named(roots, f"{split}_captions.csv"),
    ])
    errors = []
    for csv_path in csv_candidates:
        if not csv_path.is_file():
            continue
        try:
            frame = pd.read_csv(csv_path, dtype={"ID": str})
            validate_dataframe(frame)
        except Exception as error:
            errors.append(f"{csv_path}: {error}")
            continue
        ids = frame["ID"].astype(str)
        images_dir = find_images(split, ids, cfg, csv_path, roots, images_root)
        if images_dir is not None:
            return csv_path, images_dir, len(frame)
        archives = unique_paths([
            csv_path.parent / f"{split}_images.zip",
            Path(data_root) / f"{split}_images.zip",
            *files_named(roots, f"{split}_images.zip"),
        ])
        for archive in archives:
            if not archive.is_file():
                continue
            destination = Path(images_root) / f"{split}_ROCOv2"
            print(f"Extracting local archive: {archive}")
            safe_extract(archive, destination)
            images_dir = find_images(split, ids, cfg, csv_path, [destination], images_root)
            if images_dir is not None:
                return csv_path, images_dir, len(frame)
    if errors:
        print("CSV files found but not usable:", "\n".join(errors[:3]))
    return None


def prepare_data(cfg, data_root, images_root, mode="auto", search_roots=()):
    if mode not in ("auto", "local", "download"):
        raise ValueError("DATA_MODE must be 'auto', 'local', or 'download'.")
    roots = unique_paths([*search_roots, data_root, images_root])
    resolved = {}
    for split in ("train", "test"):
        local = resolve_split(split, cfg, roots, data_root, images_root)
        if local is not None:
            resolved[split] = local
    missing = [split for split in ("train", "test") if split not in resolved]
    if missing:
        missing_text = ", ".join(missing)
        help_text = (
            "Attach ROCOv2 train_captions.csv, test_captions.csv and the corresponding "
            "image folders (or train_images.zip/test_images.zip) using Kaggle Add Input. "
            "Then rerun this cell. You may set the four explicit paths in the settings cell."
        )
        if mode == "local":
            raise RuntimeError(f"Local ROCOv2 data is incomplete for: {missing_text}. {help_text}") from None
        online, reason = probe_url("https://zenodo.org/api/records/10821435")
        if not online:
            raise RuntimeError(
                f"ROCOv2 data is missing for: {missing_text}, and Zenodo cannot be reached. "
                "Enable Internet in the Kaggle notebook settings and rerun this cell, "
                f"or use DATA_MODE='local'. {help_text} Network diagnostic: {reason}"
            ) from None
        download_and_extract(data_root, images_root, splits=missing)
        for split in missing:
            cfg["paths"][f"{split}_csv"] = str(Path(data_root) / f"{split}_captions.csv")
            cfg["paths"][f"{split}_images_dir"] = str(Path(images_root) / f"{split}_ROCOv2" / split)
            local = resolve_split(split, cfg, roots, data_root, images_root)
            if local is None:
                raise RuntimeError(f"Downloaded {split} files could not be matched to CSV IDs. {help_text}")
            resolved[split] = local
    for split, (csv_path, folder, count) in resolved.items():
        cfg["paths"][f"{split}_csv"] = str(csv_path)
        cfg["paths"][f"{split}_images_dir"] = str(folder)
        print(f"{split}: {count:,} images\n  CSV: {csv_path}\n  Images: {folder}")
    return cfg


def full_model_files(directory):
    directory = Path(directory)
    if not all((directory / name).is_file() for name in ("config.json", "preprocessor_config.json")):
        return False
    if not any((directory / name).is_file() for name in ("tokenizer.json", "vocab.txt")):
        return False
    for name in ("model.safetensors", "pytorch_model.bin"):
        if (directory / name).is_file():
            return True
    for name in ("model.safetensors.index.json", "pytorch_model.bin.index.json"):
        index = directory / name
        if index.is_file():
            try:
                shards = set(json.loads(index.read_text())["weight_map"].values())
                if shards and all((directory / shard).is_file() for shard in shards):
                    return True
            except (ValueError, KeyError):
                pass
    return False


def prepare_model_source(cfg):
    source = cfg["model"]["base_model"]
    path = Path(source)
    if path.is_dir():
        if not full_model_files(path):
            raise RuntimeError(
                "MODEL_SOURCE must be a complete pretrained BLIP folder containing model weights, "
                "config.json, preprocessor_config.json and tokenizer files. A LoRA adapter alone is not enough."
            ) from None
        cfg["model"]["local_files_only"] = True
        print("Using the local pretrained BLIP model:", source)
        return cfg
    if path.is_absolute():
        raise FileNotFoundError(f"MODEL_SOURCE does not exist: {source}")
    from huggingface_hub import snapshot_download
    try:
        cached = snapshot_download(repo_id=source, local_files_only=True)
    except Exception:
        cached = None
    if cached and full_model_files(cached):
        cfg["model"]["local_files_only"] = True
        print("Using the complete cached pretrained model:", source)
        return cfg
    online, reason = probe_url(f"https://huggingface.co/api/models/{source}")
    if not online:
        raise RuntimeError(
            "BLIP weights are not fully cached and Hugging Face cannot be reached. "
            "Enable Kaggle Internet, or attach the FULL pretrained BLIP model folder and "
            f"set MODEL_SOURCE to its path. Dataset files alone do not provide model weights. Detail: {reason}"
        ) from None
    cfg["model"]["local_files_only"] = False
    print("Hugging Face is reachable; pretrained weights will be downloaded when the model is loaded.")
    return cfg
'''
module_path = CODE_ROOT / 'src/data/prepare.py'
module_path.parent.mkdir(parents=True, exist_ok=True)
module_path.write_text(source, encoding='utf-8')


8251

### `src/evaluate.py`

In [9]:
source = r'''"""Reload the saved adapter, generate captions, then save and score them."""

import argparse
from functools import partial
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import BlipForConditionalGeneration, BlipProcessor

from src.data.dataset import ImageCaptioningDataset, MergeImageAndCaption, collate_fn
from src.metrics import score_predictions
from src.models.blip_lora import load_model_and_processor
from src.utils import environment_info, get_device, load_config, seed_everything, write_json


def generate_captions(model, dataloader, processor, device, eval_cfg):
    model.eval()
    records = []
    with torch.inference_mode():
        for batch in tqdm(dataloader, desc="Generating captions"):
            output_ids = model.generate(
                pixel_values=batch["pixel_values"].to(device),
                num_beams=int(eval_cfg["num_beams"]), max_length=int(eval_cfg["max_length"]),
                do_sample=False, early_stopping=int(eval_cfg["num_beams"]) > 1,
            )
            captions = processor.batch_decode(output_ids, skip_special_tokens=True)
            records.extend(
                {"image_id": image_id, "reference": reference, "prediction": caption.strip()}
                for image_id, reference, caption in zip(batch["image_ids"], batch["texts"], captions)
            )
    return pd.DataFrame(records, columns=["image_id", "reference", "prediction"])


def evaluate_model(cfg, zero_shot=False, limit=None):
    seed_everything(int(cfg["data"]["seed"]))
    if limit is not None and limit < 1:
        raise ValueError("limit must be positive.")
    device = get_device()
    paths, eval_cfg = cfg["paths"], cfg["evaluation"]
    frame = pd.read_csv(paths["test_csv"], dtype={"ID": str})
    total_test_samples = len(frame)
    if limit is not None:
        frame = frame.head(limit).copy()
    dataset = MergeImageAndCaption(frame, paths["test_images_dir"])
    if zero_shot:
        local_only = bool(cfg["model"].get("local_files_only", False))
        model = BlipForConditionalGeneration.from_pretrained(cfg["model"]["base_model"], local_files_only=local_only)
        processor = BlipProcessor.from_pretrained(cfg["model"]["base_model"], local_files_only=local_only, use_fast=False)
    else:
        # Explicitly reconstruct the base model plus the SAVED LoRA adapter.
        model, processor = load_model_and_processor(
            paths["output_dir"], local_files_only=bool(cfg["model"].get("local_files_only", False)),
        )
    model.to(device)
    loader = DataLoader(
        ImageCaptioningDataset(dataset, processor, int(cfg["data"]["max_text_length"])),
        batch_size=int(eval_cfg.get("batch_size", cfg["training"]["batch_size"])),
        shuffle=False, num_workers=int(cfg["data"].get("num_workers", 0)),
        collate_fn=partial(collate_fn, processor=processor), pin_memory=device.type == "cuda",
    )
    result_dir = Path(eval_cfg["output_dir"]) / ("zero-shot" if zero_shot else "fine-tuned")
    if limit is not None:
        result_dir = result_dir / f"smoke-{limit}"
    result_dir.mkdir(parents=True, exist_ok=True)
    predictions = generate_captions(model, loader, processor, device, eval_cfg)
    predictions_path = result_dir / "predictions.csv"
    predictions.to_csv(predictions_path, index=False)
    write_json(result_dir / "evaluation_config.json", {
        "mode": "zero-shot" if zero_shot else "fine-tuned", "config": cfg,
        "requested_limit": limit, "full_test_samples": total_test_samples,
        "evaluated_samples": len(predictions), "is_smoke_test": limit is not None,
        "checkpoint": cfg["model"]["base_model"] if zero_shot else paths["output_dir"],
        "environment": environment_info(),
    })
    print(f"Saved {len(predictions):,} predictions to {predictions_path}")
    scores = score_predictions(predictions_path, result_dir / "metrics.json")
    return predictions, scores


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", default="configs/config.yaml")
    parser.add_argument("--zero-shot", action="store_true")
    parser.add_argument("--limit", type=int, help="Small smoke test only; not final benchmark results.")
    args = parser.parse_args()
    evaluate_model(load_config(args.config), zero_shot=args.zero_shot, limit=args.limit)


if __name__ == "__main__":
    main()
'''
module_path = CODE_ROOT / 'src/evaluate.py'
module_path.parent.mkdir(parents=True, exist_ok=True)
module_path.write_text(source, encoding='utf-8')


4448

### `src/metrics.py`

In [10]:
source = r'''"""COCO caption metrics with a single, explicit tokenization protocol."""

import argparse
import shutil
from pathlib import Path

import pandas as pd

from src.utils import write_json


def score_captions(references, predictions):
    if not references or len(references) != len(predictions):
        raise ValueError("Provide equally sized, nonempty references and predictions.")
    if shutil.which("java") is None:
        raise RuntimeError(
            "Java is required by COCO PTBTokenizer and METEOR. Install a Java runtime, "
            "then rerun scoring; the generated predictions.csv does not need to be regenerated."
        )
    from pycocoevalcap.bleu.bleu import Bleu
    from pycocoevalcap.cider.cider import Cider
    from pycocoevalcap.meteor.meteor import Meteor
    from pycocoevalcap.rouge.rouge import Rouge
    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer

    tokenizer = PTBTokenizer()
    ground_truth = tokenizer.tokenize({i: [{"caption": str(text)}] for i, text in enumerate(references)})
    generated = tokenizer.tokenize({i: [{"caption": str(text)}] for i, text in enumerate(predictions)})
    bleu, _ = Bleu(4).compute_score(ground_truth, generated)
    rouge, _ = Rouge().compute_score(ground_truth, generated)
    cider, _ = Cider().compute_score(ground_truth, generated)
    meteor_scorer = Meteor()
    meteor, _ = meteor_scorer.compute_score(ground_truth, generated)
    return {
        "BLEU-1": float(bleu[0]), "BLEU-2": float(bleu[1]),
        "BLEU-3": float(bleu[2]), "BLEU-4": float(bleu[3]),
        "METEOR": float(meteor), "ROUGE-L": float(rouge), "CIDEr": float(cider),
    }


def score_predictions(predictions_path, output_path):
    frame = pd.read_csv(predictions_path, dtype={"image_id": str}, keep_default_na=False)
    if not {"reference", "prediction"}.issubset(frame.columns):
        raise ValueError("Predictions CSV needs reference and prediction columns.")
    scores = score_captions(frame["reference"].tolist(), frame["prediction"].tolist())
    write_json(output_path, {
        "n_samples": len(frame), "scores": scores,
        "tokenization": "pycocoevalcap PTBTokenizer, same protocol for all models",
        "references": "original, untruncated CSV captions",
        "predictions_file": str(predictions_path),
    })
    for name, value in scores.items():
        print(f"{name}: {value:.4f}")
    return scores


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--predictions", required=True)
    parser.add_argument("--output")
    args = parser.parse_args()
    output = args.output or str(Path(args.predictions).with_name("metrics.json"))
    score_predictions(args.predictions, output)
'''
module_path = CODE_ROOT / 'src/metrics.py'
module_path.parent.mkdir(parents=True, exist_ok=True)
module_path.write_text(source, encoding='utf-8')


2726

### `src/models/__init__.py`

In [11]:
source = r'''"""BLIP and PEFT helpers."""
'''
module_path = CODE_ROOT / 'src/models/__init__.py'
module_path.parent.mkdir(parents=True, exist_ok=True)
module_path.write_text(source, encoding='utf-8')


29

### `src/models/blip_lora.py`

In [12]:
source = r'''"""The original BLIP + query/key LoRA configuration, with explicit reload."""

from pathlib import Path

from peft import LoraConfig, PeftConfig, PeftModel, get_peft_model
from transformers import BlipForConditionalGeneration, BlipProcessor


def count_parameters(model):
    total = sum(param.numel() for param in model.parameters())
    trainable = sum(param.numel() for param in model.parameters() if param.requires_grad)
    counts = {"total": total, "trainable": trainable, "trainable_percent": 100 * trainable / total}
    print(f"Total parameters: {total:,}")
    print(f"Trainable parameters: {trainable:,} ({counts['trainable_percent']:.3f}%)")
    return counts


def apply_lora(model, lora_cfg):
    targets = lora_cfg["target_modules"]
    matches = [name for name, _ in model.named_modules() if any(name.endswith(t) for t in targets)]
    if not matches:
        raise ValueError(f"No LoRA target modules matched: {targets}")
    model = get_peft_model(model, LoraConfig(
        r=int(lora_cfg["r"]), lora_alpha=int(lora_cfg["alpha"]),
        lora_dropout=float(lora_cfg["dropout"]), bias=lora_cfg["bias"],
        target_modules=targets,
    ))
    print(f"LoRA applied to {len(matches)} query/key projections.")
    return model


def build_model_and_processor(cfg):
    local_only = bool(cfg.get("local_files_only", False))
    processor = BlipProcessor.from_pretrained(cfg["base_model"], local_files_only=local_only, use_fast=False)
    model = BlipForConditionalGeneration.from_pretrained(cfg["base_model"], local_files_only=local_only)
    return apply_lora(model, cfg["lora"]), processor


def load_model_and_processor(checkpoint, local_files_only=False):
    checkpoint = Path(checkpoint)
    if not (checkpoint / "adapter_config.json").is_file():
        raise FileNotFoundError(f"No saved LoRA adapter found in {checkpoint}.")
    adapter_cfg = PeftConfig.from_pretrained(str(checkpoint))
    base = BlipForConditionalGeneration.from_pretrained(
        adapter_cfg.base_model_name_or_path, local_files_only=local_files_only,
    )
    model = PeftModel.from_pretrained(base, str(checkpoint), is_trainable=False)
    processor = BlipProcessor.from_pretrained(str(checkpoint), use_fast=False)
    return model, processor
'''
module_path = CODE_ROOT / 'src/models/blip_lora.py'
module_path.parent.mkdir(parents=True, exist_ok=True)
module_path.write_text(source, encoding='utf-8')


2245

### `src/train.py`

In [13]:
source = r'''"""Fine-tune on the same 10% / six-epoch setup after fixing preprocessing."""

import argparse
from datetime import datetime, timezone
from functools import partial
from pathlib import Path

import pandas as pd
import torch
import yaml
from torch.optim import AdamW
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import get_scheduler

from src.data.dataset import ImageCaptioningDataset, MergeImageAndCaption, collate_fn, validate_dataframe
from src.models.blip_lora import build_model_and_processor, count_parameters
from src.utils import environment_info, get_device, load_config, seed_everything, seed_worker, write_json


def training_inputs(batch, device, bos_token_id):
    # BLIP may replace the first input token in-place; never alias labels/input_ids.
    inputs = {
        "pixel_values": batch["pixel_values"].to(device),
        "input_ids": batch["input_ids"].to(device).clone(),
        "attention_mask": batch["attention_mask"].to(device),
        "labels": batch["labels"].to(device).clone(),
    }
    # BlipProcessor's BERT tokenizer emits [CLS], but caption generation starts
    # with the BLIP decoder's [DEC]/BOS token. Align training with inference.
    inputs["input_ids"][:, 0] = int(bos_token_id)
    inputs["labels"][:, 0] = int(bos_token_id)
    return inputs


def train_one_epoch(model, dataloader, optimizer, lr_scheduler, device):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0
    progress = tqdm(dataloader, desc="Training")
    for batch in progress:
        outputs = model(**training_inputs(batch, device, model.config.text_config.bos_token_id))
        loss = outputs.loss
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss. Stop and inspect the batch/configuration.")
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        total_loss += loss.detach().item()
        progress.set_postfix(loss=f"{loss.item():.4f}")
    return total_loss / len(dataloader)


def train_model(cfg):
    seed = int(cfg["data"]["seed"])
    seed_everything(seed)
    paths = cfg["paths"]
    output_dir = Path(paths["output_dir"])
    if (output_dir / "adapter_config.json").exists():
        raise FileExistsError(
            f"An adapter already exists at {output_dir}. Use a new output_dir for a fresh run. "
            "This script does not silently resume or overwrite a previous experiment."
        )
    frame = pd.read_csv(paths["train_csv"], dtype={"ID": str})
    validate_dataframe(frame)
    sample = frame.sample(frac=float(cfg["data"]["train_fraction"]), random_state=seed).reset_index(drop=True)
    if sample.empty:
        raise ValueError("The selected training fraction produced zero samples.")
    dataset = MergeImageAndCaption(sample, paths["train_images_dir"])
    model, processor = build_model_and_processor(cfg["model"])
    device = get_device()
    model.to(device)
    counts = count_parameters(model)
    generator = torch.Generator().manual_seed(seed)
    dataloader = DataLoader(
        ImageCaptioningDataset(dataset, processor, int(cfg["data"]["max_text_length"])),
        batch_size=int(cfg["training"]["batch_size"]), shuffle=True,
        collate_fn=partial(collate_fn, processor=processor),
        num_workers=int(cfg["data"].get("num_workers", 0)),
        worker_init_fn=seed_worker, generator=generator, pin_memory=device.type == "cuda",
    )
    optimizer = AdamW(
        [param for param in model.parameters() if param.requires_grad],
        lr=float(cfg["training"]["learning_rate"]),
        weight_decay=float(cfg["training"].get("weight_decay", 0.01)),
    )
    epochs = int(cfg["training"]["num_epochs"])
    scheduler = get_scheduler(
        cfg["training"]["lr_scheduler"], optimizer=optimizer,
        num_warmup_steps=int(cfg["training"]["warmup_steps"]),
        num_training_steps=len(dataloader) * epochs,
    )
    output_dir.mkdir(parents=True, exist_ok=True)
    sample[["ID"]].to_csv(output_dir / "train_sample_ids.csv", index=False)
    (output_dir / "run_config.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    write_json(output_dir / "environment.json", environment_info())
    metadata = {
        "status": "running", "started_utc": datetime.now(timezone.utc).isoformat(),
        "seed": seed, "training_samples": len(sample), "optimizer_steps_per_epoch": len(dataloader),
        "total_optimizer_steps": len(dataloader) * epochs, "parameter_counts": counts,
        "image_processor": processor.image_processor.to_dict(),
        "checkpoint_rule": "last completed epoch; no validation-based selection",
        "precision": "float32", "completed_epochs": 0,
    }
    write_json(output_dir / "training_state.json", metadata)
    history = []
    print(f"Device: {device}; training images: {len(sample):,}; epochs: {epochs}")
    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")
        loss = train_one_epoch(model, dataloader, optimizer, scheduler, device)
        history.append({"epoch": epoch + 1, "train_loss": loss, "learning_rate": scheduler.get_last_lr()[0]})
        pd.DataFrame(history).to_csv(output_dir / "history.csv", index=False)
        model.save_pretrained(str(output_dir), safe_serialization=True)
        processor.save_pretrained(str(output_dir))
        torch.save(optimizer.state_dict(), output_dir / "optimizer_state.pt")
        torch.save(scheduler.state_dict(), output_dir / "scheduler_state.pt")
        metadata["completed_epochs"] = epoch + 1
        metadata["status"] = "completed" if epoch + 1 == epochs else "running"
        write_json(output_dir / "training_state.json", metadata)
        print(f"Training loss: {loss:.4f}; adapter saved to {output_dir}")
    return model, processor, history


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", default="configs/config.yaml")
    args = parser.parse_args()
    train_model(load_config(args.config))


if __name__ == "__main__":
    main()
'''
module_path = CODE_ROOT / 'src/train.py'
module_path.parent.mkdir(parents=True, exist_ok=True)
module_path.write_text(source, encoding='utf-8')


6103

### `src/utils.py`

In [14]:
source = r'''"""Configuration, reproducibility, and run metadata."""

import importlib.metadata
import json
import platform
import random
from pathlib import Path

import numpy as np
import torch
import yaml


def load_config(path="configs/config.yaml"):
    with open(path, encoding="utf-8") as handle:
        cfg = yaml.safe_load(handle)
    if not 0 < float(cfg["data"]["train_fraction"]) <= 1:
        raise ValueError("data.train_fraction must be in (0, 1].")
    if int(cfg["training"]["batch_size"]) < 1 or int(cfg["training"]["num_epochs"]) < 1:
        raise ValueError("batch_size and num_epochs must be positive.")
    if not 2 <= int(cfg["data"]["max_text_length"]) <= 512:
        raise ValueError("BLIP max_text_length must be between 2 and 512.")
    return cfg


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")


def environment_info():
    packages = {}
    for name in ("torch", "transformers", "peft", "accelerate", "numpy", "pandas", "pycocoevalcap"):
        try:
            packages[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            packages[name] = None
    return {
        "python": platform.python_version(),
        "packages": packages,
        "cuda_available": torch.cuda.is_available(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }
'''
module_path = CODE_ROOT / 'src/utils.py'
module_path.parent.mkdir(parents=True, exist_ok=True)
module_path.write_text(source, encoding='utf-8')


1998

## 5. Experiment Configuration

Image preprocessing follows the pretrained model's processor configuration. Training captions are capped at 512 tokens, and padding positions are excluded from the loss. Complete original reference captions are retained for evaluation.


In [15]:
from IPython.display import display
import copy
import importlib
import json
import shutil
import torch
import transformers
import peft
import yaml

importlib.invalidate_caches()
from src.utils import environment_info

cfg = {
    "paths": {
        "train_images_dir": str(IMAGES_ROOT / "train_ROCOv2" / "train"),
        "test_images_dir": str(IMAGES_ROOT / "test_ROCOv2" / "test"),
        "train_csv": str(DATA_ROOT / "train_captions.csv"),
        "test_csv": str(DATA_ROOT / "test_captions.csv"),
        "output_dir": str(WORK_ROOT / "checkpoints" / RUN_NAME),
    },
    "data": {"train_fraction": 0.10, "seed": 42, "max_text_length": 512, "num_workers": 0},
    "model": {
        "base_model": "Salesforce/blip-image-captioning-base",
        "lora": {"r": 8, "alpha": 16, "dropout": 0.1, "bias": "none",
                 "target_modules": ["attention.self.query", "attention.self.key"]},
    },
    "training": {"batch_size": 3, "num_epochs": 6, "learning_rate": 5e-5,
                 "weight_decay": 0.01, "lr_scheduler": "linear", "warmup_steps": 0},
    "evaluation": {"batch_size": 3, "num_beams": 4, "max_length": 128,
                   "output_dir": str(WORK_ROOT / "results" / RUN_NAME)},
}
config_path = CODE_ROOT / "configs" / "config.yaml"
config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print(json.dumps(environment_info(), indent=2))
print("Java available:", shutil.which("java") is not None)
if shutil.which("java") is None:
    print("Java is needed only for scoring. Use the optional Java installation cell before final metrics.")
print("Checkpoint:", cfg["paths"]["output_dir"])
if not torch.cuda.is_available() and RUN_TRAINING:
    print("WARNING: no GPU detected. Select a GPU before the full training cell.")

# Apply user-selected model source and optional explicit data paths.
cfg["model"]["base_model"] = MODEL_SOURCE
overrides = {
    "train_csv": LOCAL_TRAIN_CSV,
    "test_csv": LOCAL_TEST_CSV,
    "train_images_dir": LOCAL_TRAIN_IMAGES_DIR,
    "test_images_dir": LOCAL_TEST_IMAGES_DIR,
}
for key, value in overrides.items():
    if value is not None:
        cfg["paths"][key] = str(value)


{
  "python": "3.13.15",
  "packages": {
    "torch": "2.11.0+cu128",
    "transformers": "4.57.6",
    "peft": "0.18.1",
    "accelerate": "1.12.0",
    "numpy": "2.1.3",
    "pandas": "2.3.3",
    "pycocoevalcap": "1.2"
  },
  "cuda_available": true,
  "gpu": "Tesla T4"
}
Java available: True
Checkpoint: /kaggle/working/checkpoints/blip-corrected-network-v2-10pct-6epochs


## 6. Prepare Data and Model — Local Files First

The notebook first checks access to the pretrained model or its complete local cache. It then searches for captions and existing images or local image archives. Only missing splits are downloaded, and only when Zenodo is reachable. A fresh download contains approximately 5.5 GB of compressed files; extraction requires additional disk space.

If access fails, enable Internet in Kaggle and rerun this cell. Alternatively, attach the complete dataset and model using **Add Input**, enter their paths in the settings cell, select local mode, and rerun the settings and experiment configuration cells before retrying.

Transient network failures receive a limited number of retries. Repeated requests cannot enable disabled Internet access or repair the runtime's DNS configuration.


In [16]:
from src.data.prepare import prepare_data, prepare_model_source

# Check model access/cache first, before any multi-GB dataset download.
cfg = prepare_model_source(cfg)

# Resolve complete local folders or attached ZIPs before contacting Zenodo.
cfg = prepare_data(
    cfg,
    data_root=DATA_ROOT,
    images_root=IMAGES_ROOT,
    mode=DATA_MODE,
    search_roots=SEARCH_ROOTS,
)
config_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print("Data and model preflight passed.")


Hugging Face is reachable; pretrained weights will be downloaded when the model is loaded.


train_captions.csv:   0%|          | 0.00/10.1M [00:00<?, ?B/s]

train_images.zip:   0%|          | 0.00/4.61G [00:00<?, ?B/s]

Extracting train_images.zip ...


test_captions.csv:   0%|          | 0.00/1.77M [00:00<?, ?B/s]

test_images.zip:   0%|          | 0.00/858M [00:00<?, ?B/s]

Extracting test_images.zip ...
Data preparation complete. Check the CSV/image paths in your configuration.
train: 59,958 images
  CSV: /kaggle/working/train_captions.csv
  Images: /kaggle/working/datasets/train_ROCOv2/train
test: 9,927 images
  CSV: /kaggle/working/test_captions.csv
  Images: /kaggle/working/datasets/test_ROCOv2/test
Data and model preflight passed.


## 7. One-Batch Forward/Backward Sanity Check

This uses two training images and the actual pretrained BLIP model. It checks finite loss and LoRA gradients, without an optimizer step or checkpoint update. Its temporary model is then freed before the fresh training run. A failure should be fixed before starting all six epochs.


In [17]:
import gc
import pandas as pd
from src.data.dataset import MergeImageAndCaption, ImageCaptioningDataset, collate_fn
from src.models.blip_lora import build_model_and_processor, count_parameters
from src.train import training_inputs
from src.utils import get_device, seed_everything

def check_one_batch(config):
    seed_everything(config["data"]["seed"])
    frame = pd.read_csv(config["paths"]["train_csv"], dtype={"ID": str}).head(2)
    raw = MergeImageAndCaption(frame, config["paths"]["train_images_dir"])
    sanity_model, sanity_processor = build_model_and_processor(config["model"])
    dataset = ImageCaptioningDataset(raw, sanity_processor, config["data"]["max_text_length"])
    batch = collate_fn([dataset[i] for i in range(len(dataset))], sanity_processor)
    device = get_device()
    sanity_model.to(device).train()
    output = sanity_model(**training_inputs(batch, device, sanity_model.config.text_config.bos_token_id))
    assert torch.isfinite(output.loss), "Loss is not finite."
    output.loss.backward()
    assert any(p.grad is not None and torch.isfinite(p.grad).all() and p.grad.abs().sum() > 0
               for name, p in sanity_model.named_parameters() if "lora_" in name), "No finite nonzero LoRA gradient."
    print("pixel_values:", tuple(batch["pixel_values"].shape))
    print("input_ids:", tuple(batch["input_ids"].shape))
    print("Ignored padding labels:", int((batch["labels"] == -100).sum()))
    print("Sanity-check loss:", float(output.loss.detach()))
    count_parameters(sanity_model)
    print("Forward/backward sanity check passed; no optimizer update was performed.")

if RUN_SANITY_CHECK:
    check_one_batch(cfg)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

LoRA applied to 48 query/key projections.
pixel_values: (2, 3, 384, 384)
input_ids: (2, 36)
Ignored padding labels: 25
Sanity-check loss: 7.79864501953125
Total parameters: 224,266,556
Trainable parameters: 294,912 (0.132%)
Forward/backward sanity check passed; no optimizer update was performed.


## 8. Train from Pretrained Weights

Fine-tuning starts from the original pretrained BLIP checkpoint with a new LoRA adapter. It does not continue the previous buggy run. Test data is not used here. The last completed epoch is saved; this fixed six-epoch experiment does not use validation-based model selection.


In [18]:
from src.train import train_model

if RUN_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU detected. Select a Kaggle GPU runtime before full training.") from None
    trained_model, trained_processor, history = train_model(copy.deepcopy(cfg))
    display(pd.DataFrame(history))
    # Evaluation reloads the saved adapter. Free this model first to avoid holding two on GPU.
    del trained_model, trained_processor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("Training skipped. Evaluation requires a completed corrected checkpoint at output_dir.")


LoRA applied to 48 query/key projections.
Total parameters: 224,266,556
Trainable parameters: 294,912 (0.132%)
Device: cuda; training images: 5,996; epochs: 6
Epoch 1/6


Training:   0%|          | 0/1999 [00:00<?, ?it/s]

Training loss: 6.4668; adapter saved to /kaggle/working/checkpoints/blip-corrected-network-v2-10pct-6epochs
Epoch 2/6


Training:   0%|          | 0/1999 [00:00<?, ?it/s]

Training loss: 6.1454; adapter saved to /kaggle/working/checkpoints/blip-corrected-network-v2-10pct-6epochs
Epoch 3/6


...

Training:   0%|          | 0/1999 [00:00<?, ?it/s]

Training loss: 5.9150; adapter saved to /kaggle/working/checkpoints/blip-corrected-network-v2-10pct-6epochs
Epoch 6/6


Training:   0%|          | 0/1999 [00:00<?, ?it/s]

Training loss: 5.9045; adapter saved to /kaggle/working/checkpoints/blip-corrected-network-v2-10pct-6epochs


,epoch,train_loss,learning_rate
0,1,6.466787,0.000042
1,2,6.145403,0.000033
2,3,6.033801,0.000025
3,4,5.968374,0.000017
4,5,5.915012,0.000008
5,6,5.904488,0.000000


## 9. Java for Caption Metrics (Optional)

If Java is not installed, set `INSTALL_JAVA=True` and run this cell. Installation requires Internet access and the appropriate runtime permissions.

If Java installation is unavailable, caption generation still saves `predictions.csv` before scoring. You can calculate the metrics later from that CSV without repeating model inference.


In [19]:
import subprocess
INSTALL_JAVA = False
if INSTALL_JAVA and shutil.which("java") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "default-jre-headless"], check=True)
print("Java:", shutil.which("java"))


Java: /usr/bin/java


## 10. Evaluate the Saved Adapter on the Full Test Set

This explicitly reloads BLIP plus the saved LoRA adapter and processor, rather than evaluating only an in-memory training object. Predictions are saved before metrics. Java is required for PTBTokenizer/METEOR; if scoring fails, fix Java and run the next cell to rescore without repeating generation.

For a quick three-image inference check only, temporarily use `limit=3`. Those scores are not a full benchmark and are saved in a separate smoke-test folder. Use `limit=None` for final evaluation. Do not tune repeatedly on test results.


In [20]:
from src.evaluate import evaluate_model

if RUN_EVALUATION:
    predictions, scores = evaluate_model(copy.deepcopy(cfg), zero_shot=False, limit=None)
    display(predictions.head(5))


Generating captions:   0%|          | 0/3309 [00:00<?, ?it/s]

Saved 9,927 predictions to /kaggle/working/results/blip-corrected-network-v2-10pct-6epochs/fine-tuned/predictions.csv


Oct 17, 2026 7:07:55 AM edu.stanford.nlp.process.PTBLexer next
PTBTokenizer tokenized 265178 tokens at 365131.79 tokens per second.
PTBTokenizer tokenized 226729 tokens at 595330.53 tokens per second.


{'testlen': 206227, 'reflen': 233389, 'guess': [206227, 196300, 186373, 176446], 'correct': [34360, 6582, 561, 67]}
ratio: 0.8836191937066405
BLEU-1: 0.1461
BLEU-2: 0.0655
BLEU-3: 0.0225
BLEU-4: 0.0078
METEOR: 0.0475
ROUGE-L: 0.1302
CIDEr: 0.0351


,image_id,reference,prediction
0,ROCOv2_2023_test_000001,CT chest axial view showing a huge ascending a...,the ct scan shows a small portion of the brain...
1,ROCOv2_2023_test_000002,Computed tomography (CT) shows floating thromb...,an image of a bird on a black background
2,ROCOv2_2023_test_000003,Digitally subtracted angiogram demonstrates ac...,an image of the left side of the left side of ...
3,ROCOv2_2023_test_000004,Digitally subtracted angiogram of the IMA demo...,an image of the surface of the surface of the ...
4,ROCOv2_2023_test_000005,Angle measurement of a Type 1 canal.,the right side of the right side of the right ...


## 11. Rescore Saved Captions (Optional)

Use this if generation completed but Java/scoring failed, or to inspect the saved CSV. No new model inference is needed. Leave the switch off after a successful full evaluation.


In [21]:
from src.metrics import score_predictions

RESCORE_SAVED_CSV = False
if RESCORE_SAVED_CSV:
    result_dir = Path(cfg["evaluation"]["output_dir"]) / "fine-tuned"
    rescored = score_predictions(result_dir / "predictions.csv", result_dir / "metrics.json")


## 12. Zero-Shot Baseline (Optional)

This evaluates the original BLIP checkpoint on the same test CSV and with the same image processing, generation settings and metric tokenization. Results go into a separate `zero-shot` folder. It is not a comparison against the old notebook's metrics.


In [22]:
if RUN_ZERO_SHOT_BASELINE:
    baseline_predictions, baseline_scores = evaluate_model(copy.deepcopy(cfg), zero_shot=True, limit=None)
    display(baseline_predictions.head(5))


## 13. Output Files and Verification

- **Checkpoint directory:** LoRA adapter, processor, experiment configuration, sampled training IDs, environment details, training state, and loss history.
- **Result directory:** `predictions.csv`, `evaluation_config.json`, and `metrics.json`.

Download these directories from your notebook outputs after the run. Update historical README scores only after completing the corrected experiment and inspecting actual generated captions. Code corrections alone do not guarantee better caption quality.

### Troubleshooting the Reported Error

`gaierror: [Errno -3] Temporary failure in name resolution` means the runtime could not resolve a hostname. Possible causes include disabled Internet access or a temporary DNS failure. Local data mode avoids Zenodo requests, but installing packages and loading uncached model weights still requires Internet access or previously prepared local files.

### Verification Scope

The tiny-model CPU pipeline was checked with Transformers 4.44.2 / PEFT 0.12.0 and with Transformers 4.57.6 / PEFT 0.18.1. The newer profile was tested on Python 3.12.14. Python 3.13 and GPU execution were not tested in that environment. Actual package versions are recorded in `environment.json`.

### Official References

- [ROCOv2 dataset](https://zenodo.org/records/10821435)
- [BLIP documentation](https://huggingface.co/docs/transformers/v4.44.2/en/model_doc/blip)
- [PEFT documentation](https://huggingface.co/docs/peft/v0.12.0/en/quicktour)
- [COCO caption evaluation](https://github.com/salaniz/pycocoevalcap)
